In [0]:
%pip install openpyxl

In [0]:
# Defino o caminho do arquivo original que armazenei na camada RAW.
caminho_arquivo = """
/Volumes/workspace/raw/anp_arquivos/anp/f40da41fecc036126f442d6aa0e1bb423b7261aff8d8ea47fbe701b3c1ac778b/revendas_lpc_2026-09-06_2026-09-12.xlsx
""".strip()

print(caminho_arquivo)

In [0]:
import os

# Antes de processar os dados, confirmo que o arquivo está acessível no Volume.
arquivo_encontrado = os.path.exists(caminho_arquivo)

print("Arquivo encontrado:", arquivo_encontrado)

In [0]:
import pandas as pd

# Leio apenas algumas linhas para validar a planilha e o cabeçalho
# antes de fazer a carga completa da camada RAW.
amostra = pd.read_excel(
    caminho_arquivo,
    sheet_name="POSTOS REVENDEDORES",
    header=9,
    nrows=5,
    engine="openpyxl"
)

display(amostra)

In [0]:
# Crio o schema que vai representar a camada Bronze do projeto.
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.bronze
    COMMENT 'Camada Bronze com os dados estruturados da ANP antes das regras de qualidade e transformação.'
""")

print("Schema Bronze criado ou já existente.")

In [0]:
# Confirmo que o schema está disponível no Unity Catalog.
display(
    spark.sql("SHOW SCHEMAS IN workspace")
)

In [0]:
import pandas as pd

# Mantenho como texto os campos que funcionam como identificadores ou descrições.
# Isso evita que CNPJ, CEP e outros valores sejam tratados como números.
colunas_texto = [
    "CNPJ",
    "RAZÃO",
    "FANTASIA",
    "ENDEREÇO",
    "NÚMERO",
    "COMPLEMENTO",
    "BAIRRO",
    "CEP",
    "MUNICÍPIO",
    "ESTADO",
    "BANDEIRA",
    "PRODUTO",
    "UNIDADE DE MEDIDA"
]

tipos_colunas = {
    coluna: "string"
    for coluna in colunas_texto
}

# Agora faço a leitura completa do arquivo original armazenado na RAW.
dados_bronze = pd.read_excel(
    caminho_arquivo,
    sheet_name="POSTOS REVENDEDORES",
    header=9,
    dtype=tipos_colunas,
    engine="openpyxl"
)

# Garanto explicitamente os tipos das duas colunas de negócio restantes.
dados_bronze["PREÇO DE REVENDA"] = pd.to_numeric(
    dados_bronze["PREÇO DE REVENDA"],
    errors="coerce"
)

dados_bronze["DATA DA COLETA"] = pd.to_datetime(
    dados_bronze["DATA DA COLETA"],
    errors="coerce"
)

print("Registros carregados:", len(dados_bronze))
print("Colunas:", len(dados_bronze.columns))

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType
)

# Defino explicitamente o schema para não depender da inferência automática
# dos tipos durante a conversão para Spark.
esquema_bronze = StructType([
    StructField("CNPJ", StringType(), True),
    StructField("RAZÃO", StringType(), True),
    StructField("FANTASIA", StringType(), True),
    StructField("ENDEREÇO", StringType(), True),
    StructField("NÚMERO", StringType(), True),
    StructField("COMPLEMENTO", StringType(), True),
    StructField("BAIRRO", StringType(), True),
    StructField("CEP", StringType(), True),
    StructField("MUNICÍPIO", StringType(), True),
    StructField("ESTADO", StringType(), True),
    StructField("BANDEIRA", StringType(), True),
    StructField("PRODUTO", StringType(), True),
    StructField("UNIDADE DE MEDIDA", StringType(), True),
    StructField("PREÇO DE REVENDA", DoubleType(), True),
    StructField("DATA DA COLETA", TimestampType(), True)
])

# Troco os valores ausentes do Pandas por None antes de enviar os dados ao Spark.
dados_para_spark = (
    dados_bronze
    .astype(object)
    .where(pd.notna(dados_bronze), None)
)

df_bronze = spark.createDataFrame(
    dados_para_spark,
    schema=esquema_bronze
)

print("Registros no Spark:", df_bronze.count())

In [0]:
from uuid import uuid4

from pyspark.sql.functions import (
    current_timestamp,
    lit
)

# Cada processamento no Databricks recebe seu próprio identificador.
id_execucao = str(uuid4())

id_arquivo = 1
arquivo_origem = "revendas_lpc_2026-09-06_2026-09-12.xlsx"

# Acrescento os metadados necessários para rastrear a origem desta carga.
df_bronze = (
    df_bronze
    .withColumn("id_execucao", lit(id_execucao))
    .withColumn("id_arquivo", lit(id_arquivo))
    .withColumn("arquivo_origem", lit(arquivo_origem))
    .withColumn("data_ingestao_utc", current_timestamp())
)

print("ID da execução:", id_execucao)
print("Registros:", df_bronze.count())
print("Colunas:", len(df_bronze.columns))

In [0]:
df_bronze.printSchema()

display(
    df_bronze.limit(10)
)

In [0]:
from pyspark.sql.functions import lit

# Uso o mesmo hash calculado durante a ingestão do arquivo original.
# Ele será a referência para impedir que o mesmo arquivo seja carregado duas vezes.
hash_sha256 = "f40da41fecc036126f442d6aa0e1bb423b7261aff8d8ea47fbe701b3c1ac778b"

df_bronze = df_bronze.withColumn(
    "hash_sha256",
    lit(hash_sha256)
)

print("Registros:", df_bronze.count())
print("Colunas:", len(df_bronze.columns))

In [0]:
nome_tabela_bronze = "workspace.bronze.revendas_anp"

# Crio uma visão temporária apenas para materializar a primeira versão
# da tabela Bronze no Unity Catalog.
df_bronze.createOrReplaceTempView("vw_carga_bronze_anp")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {nome_tabela_bronze}
    USING DELTA
    TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name'
    )
    AS
    SELECT *
    FROM vw_carga_bronze_anp
""")

print("Tabela Bronze criada ou já existente.")

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS registros,
            COUNT(DISTINCT hash_sha256) AS arquivos_distintos
        FROM workspace.bronze.revendas_anp
    """)
)

In [0]:
import hashlib


def calcular_hash_sha256(caminho):
    """
    Calculo o SHA-256 do arquivo em blocos para não precisar
    carregar todo o conteúdo na memória de uma única vez.
    """
    hash_arquivo = hashlib.sha256()

    with open(caminho, "rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            hash_arquivo.update(bloco)

    return hash_arquivo.hexdigest()


hash_calculado = calcular_hash_sha256(caminho_arquivo)

print("SHA-256 calculado:")
print(hash_calculado)

In [0]:
from pyspark.sql.functions import col, lit

nome_tabela_bronze = "workspace.bronze.revendas_anp"

# Consulto a própria Bronze para descobrir se esse arquivo
# já foi carregado anteriormente.
arquivo_ja_processado = (
    spark.table(nome_tabela_bronze)
    .filter(col("hash_sha256") == hash_calculado)
    .limit(1)
    .count()
    > 0
)

if arquivo_ja_processado:
    print("Arquivo já processado.")
    print("Carga Bronze ignorada para evitar duplicidade.")

else:
    # Uso sempre o hash calculado diretamente do arquivo RAW.
    df_bronze_carga = df_bronze.withColumn(
        "hash_sha256",
        lit(hash_calculado)
    )

    (
        df_bronze_carga
        .write
        .mode("append")
        .saveAsTable(nome_tabela_bronze)
    )

    print("Nova carga adicionada à Bronze.")

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS registros,
            COUNT(DISTINCT hash_sha256) AS arquivos_distintos
        FROM workspace.bronze.revendas_anp
    """)
)